# Predictive Modeling Pipeline — Facebook Performance Metrics

**Capstone Project Part 2**  
Predicting social media post engagement for a cosmetics brand using machine learning.

---
## 1. Problem Framing

### Business Problem

A renowned cosmetics brand wants to predict how well a Facebook post will perform *before* publishing it. By forecasting **Total Interactions** (likes + comments + shares), the social media team can optimize post characteristics (type, timing, category, paid promotion) to maximize engagement and brand building.

### Problem Type: **Regression**

The target variable (`Total Interactions`) is a **continuous numerical value**, making this a regression problem — not a classification problem.

### Target Column

**`Total Interactions`** — sum of `comment` + `like` + `share` per post.

### Features (X)

All columns known *prior to post publication*:
- `Page total likes` — number of fans at time of posting
- `Type` — post type (Photo, Status, Video, Link)
- `Category` — 1, 2, or 3
- `Post Month` — month of the year
- `Post Weekday` — day of the week
- `Post Hour` — hour of the day
- `Paid` — whether the post was promoted (0/1)

### Target (y)

**`Total Interactions`** — a single composite engagement score.

---
## 2. Data Loading

In [7]:
# Import core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries imported successfully.')

Libraries imported successfully.


In [8]:
# Load dataset
df = pd.read_csv('dataset_Facebook.csv', sep=';')

# Display first 5 rows
print('=== FIRST 5 ROWS ===')
df.head()

=== FIRST 5 ROWS ===


,Page total likes,Type,Category,Post Month,Post Weekday,Post Hour,Paid,Lifetime Post Total Reach,Lifetime Post Total Impressions,Lifetime Engaged Users,Lifetime Post Consumers,Lifetime Post Consumptions,Lifetime Post Impressions by people who have liked your Page,Lifetime Post reach by people who like your Page,Lifetime People who have liked your Page and engaged with your post,comment,like,share,Total Interactions
0,139441,Photo,2,12,4,3,0.00,2752,5091,178,109,159,3078,1640,119,4,79.00,17.00,100
1,139441,Status,2,12,3,10,0.00,10460,19057,1457,1361,1674,11710,6112,1108,5,130.00,29.00,164
2,139441,Photo,3,12,3,3,0.00,2413,4373,177,113,154,2812,1503,132,0,66.00,14.00,80
3,139441,Photo,2,12,2,10,1.00,50128,87991,2211,790,1119,61027,32048,1386,58,1572.00,147.00,1777
4,139441,Photo,2,12,2,3,0.00,7244,13594,671,410,580,6228,3200,396,19,325.00,49.00,393


In [9]:
# Dataset shape
print(f'Dataset shape: {df.shape}')
print(f'Number of rows: {df.shape[0]}')
print(f'Number of columns: {df.shape[1]}')

Dataset shape: (500, 19)
Number of rows: 500
Number of columns: 19


In [10]:
# Data types
print('=== DATA TYPES ===')
df.dtypes

=== DATA TYPES ===


Page total likes                                                         int64
Type                                                                    object
Category                                                                 int64
Post Month                                                               int64
Post Weekday                                                             int64
Post Hour                                                                int64
Paid                                                                   float64
Lifetime Post Total Reach                                                int64
Lifetime Post Total Impressions                                          int64
Lifetime Engaged Users                                                   int64
Lifetime Post Consumers                                                  int64
Lifetime Post Consumptions                                               int64
Lifetime Post Impressions by people who have liked y

In [11]:
# Missing values
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_percent})
missing_df[missing_df['Missing Count'] > 0]

=== MISSING VALUES ===


,Missing Count,Percentage
Paid,1,0.20
like,1,0.20
share,4,0.80


In [12]:
# Duplicate values
print(f'Number of duplicate rows: {df.duplicated().sum()}')

Number of duplicate rows: 0


---
## 3. Data Cleaning

In [13]:
# Make a working copy
df_clean = df.copy()

# Display missing value counts per column
print('=== MISSING VALUES PER COLUMN ===')
print(df_clean.isnull().sum())

=== MISSING VALUES PER COLUMN ===
Page total likes                                                       0
Type                                                                   0
Category                                                               0
Post Month                                                             0
Post Weekday                                                           0
Post Hour                                                              0
Paid                                                                   1
Lifetime Post Total Reach                                              0
Lifetime Post Total Impressions                                        0
Lifetime Engaged Users                                                 0
Lifetime Post Consumers                                                0
Lifetime Post Consumptions                                             0
Lifetime Post Impressions by people who have liked your Page           0
Lifetime Post rea

In [14]:
# Fill missing values in comment, like, share with 0 (no interaction means 0)
for col in ['comment', 'like', 'share']:
    df_clean[col] = df_clean[col].fillna(0).astype(int)

# Fill missing Post Hour with median
median_hour = df_clean['Post Hour'].median()
df_clean['Post Hour'] = df_clean['Post Hour'].fillna(median_hour)

# Fill missing Paid with 0 (most common value - unpaid)
df_clean['Paid'] = df_clean['Paid'].fillna(0)

# Recalculate Total Interactions after filling missing values
df_clean['Total Interactions'] = df_clean['comment'] + df_clean['like'] + df_clean['share']

print('Missing values handled.')
print(f'Remaining missing values: {df_clean.isnull().sum().sum()}')

Missing values handled.
Remaining missing values: 0


In [15]:
# Check for empty string / whitespace values in object columns
for col in df_clean.columns:
    if df_clean[col].dtype == 'object':
        blanks = df_clean[col].astype(str).str.strip().eq('').sum()
        if blanks > 0:
            print(f"Column '{col}' has {blanks} blank/empty values")

### Creating a Categorical Column via Binning

The dataset already has a categorical column (`Type`), but the requirements state:  
*"If there are no categorical columns, create one by binning one numerical feature into 3–5 fixed categories."*

Since we **do** have categorical columns (`Type`), this step is technically not required.  
However, to demonstrate the concept and potentially improve model performance, I will bin `Page total likes` into 3 categories:
- **Small** (≤ 100,000)
- **Medium** (100,001 – 125,000)
- **Large** (> 125,000)

This creates an **ordinal categorical feature** from a numerical one.

In [ ]:
# Bin Page total likes into 3 fixed categories
bins = [0, 100000, 125000, df_clean['Page total likes'].max()]
labels = ['Small', 'Medium', 'Large']
df_clean['Page_Likes_Category'] = pd.cut(
    df_clean['Page total likes'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

print('=== PAGE LIKES CATEGORY DISTRIBUTION ===')
print(df_clean['Page_Likes_Category'].value_counts().sort_index())

=== PAGE LIKES CATEGORY DISTRIBUTION ===
Page_Likes_Category
Small      62
Medium    139
Large     299
Name: count, dtype: int64


In [ ]:
# Verify no missing values remain
print('=== FINAL MISSING VALUE CHECK ===')
print(df_clean.isnull().sum())

In [ ]:
# Drop duplicate rows if any
df_clean = df_clean.drop_duplicates()
print(f'Shape after removing duplicates: {df_clean.shape}')

---
## 4. Train Test Split

**Why split before encoding/scaling?**  
Splitting *before* any preprocessing prevents **data leakage**. Data leakage occurs when information from the test set influences the training data. For example:
- If we fit `StandardScaler` on the full dataset, the mean and std are computed using test data — the model 'sees' test distribution during training.
- Similarly, fitting `OneHotEncoder` on the full dataset could expose test-category frequencies.

By splitting first, we ensure the test set remains **completely unseen** until final evaluation — giving an honest measure of generalization performance.

In [ ]:
from sklearn.model_selection import train_test_split

# Define feature columns (known before posting) and target
feature_cols = [
    'Page total likes',
    'Type',
    'Category',
    'Post Month',
    'Post Weekday',
    'Post Hour',
    'Paid',
    'Page_Likes_Category'
]

target_col = 'Total Interactions'

X = df_clean[feature_cols]
y = df_clean[target_col]

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape: {y.shape}')
print(f'\nFeatures:\n{X.dtypes}')
print(f'\nTarget range: {y.min()} to {y.max()}')

In [ ]:
# Stratify by Type to preserve minority class representation in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=X['Type']
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'\nTraining Type distribution:')
print(X_train['Type'].value_counts())
print(f'\nTest Type distribution:')
print(X_test['Type'].value_counts())

---
## 5. Feature Engineering

### Identifying Categorical vs. Numerical Columns

| Column | Type | Encoding |
|--------|------|----------|
| `Type` | Nominal | OneHotEncoder |
| `Category` | Ordinal | OrdinalEncoder |
| `Page_Likes_Category` | Ordinal | OrdinalEncoder |
| `Page total likes` | Numerical | StandardScaler |
| `Post Month` | Numerical (cyclical) | StandardScaler |
| `Post Weekday` | Numerical (cyclical) | StandardScaler |
| `Post Hour` | Numerical (cyclical) | StandardScaler |
| `Paid` | Binary | Already 0/1 → scale |

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Define column groups
nominal_cols = ['Type']
ordinal_cols = ['Category', 'Page_Likes_Category']
numerical_cols = ['Page total likes', 'Post Month', 'Post Weekday', 'Post Hour', 'Paid']

print('=== COLUMN GROUPS ===')
print(f'Nominal columns: {nominal_cols}')
print(f'Ordinal columns: {ordinal_cols}')
print(f'Numerical columns: {numerical_cols}')

In [ ]:
# Build ColumnTransformer with all preprocessing steps
# OneHotEncoder for nominal, OrdinalEncoder for ordinal, StandardScaler for numerical
preprocessor = ColumnTransformer(
    transformers=[
        ('nominal', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), nominal_cols),
        ('ordinal', OrdinalEncoder(), ordinal_cols),
        ('numerical', StandardScaler(), numerical_cols)
    ]
)

# Fit ONLY on training data (CRITICAL for preventing data leakage)
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the fitted preprocessor (NO fitting again)
X_test_processed = preprocessor.transform(X_test)

# Get feature names from the preprocessor
ohe_feature_names = preprocessor.named_transformers_['nominal'].get_feature_names_out(nominal_cols)
all_feature_names = list(ohe_feature_names) + ordinal_cols + numerical_cols

# Convert to DataFrames for inspection
X_train_processed_df = pd.DataFrame(X_train_processed, columns=all_feature_names, index=X_train.index)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=all_feature_names, index=X_test.index)

print(f'Processed training shape: {X_train_processed_df.shape}')
print(f'Processed test shape: {X_test_processed_df.shape}')
print(f'\nFeature columns ({len(all_feature_names)}):')
print(all_feature_names)
print(f'\nNaN values in train: {X_train_processed_df.isnull().sum().sum()}')
print(f'NaN values in test:  {X_test_processed_df.isnull().sum().sum()}')

In [ ]:
# Display first few rows of preprocessed training data
print('=== PREPROCESSED TRAINING DATA (first 5 rows) ===')
X_train_processed_df.head()

---
## 6. Scaling

**Why StandardScaler?**  
StandardScaler standardizes features by removing the mean and scaling to unit variance. This is important because:
- Features like `Page total likes` (~100,000) have much larger magnitudes than `Post Hour` (0–23).
- Models like Linear Regression and Gradient Boosting are sensitive to feature scales.
- StandardScaler is robust to moderate outliers and works well with normally distributed data.

**Fit only on training data** to prevent data leakage.

Note: Scaling is already integrated in the ColumnTransformer above via the StandardScaler for numerical columns.

In [ ]:
# Verify scaling was applied correctly
print('=== SCALING VERIFICATION ===')
print('Training set mean (should be ~0):')
print(X_train_processed_df[numerical_cols].mean().round(4))
print('\nTraining set std (should be ~1):')
print(X_train_processed_df[numerical_cols].std().round(4))

---
## 7. Train Models

Since this is a **Regression** problem, we train:
1. **Linear Regression** — baseline
2. **Decision Tree Regressor** — interpretable
3. **Random Forest Regressor** — ensemble (best for this dataset)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Dictionary to store results
results = {}

In [ ]:
# Check for NaN before training
print(f'NaN in X_train_processed: {np.isnan(X_train_processed).sum()}')
print(f'NaN in X_test_processed:  {np.isnan(X_test_processed).sum()}')
print(f'NaN in y_train:           {y_train.isnull().sum()}')
print(f'NaN in y_test:            {y_test.isnull().sum()}')

In [ ]:
# Model 1: Linear Regression
print('=' * 60)
print('MODEL 1: LINEAR REGRESSION')
print('=' * 60)

lr = LinearRegression()
lr.fit(X_train_processed, y_train)
y_pred_lr = lr.predict(X_test_processed)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = float(np.sqrt(mean_squared_error(y_test, y_pred_lr)))
r2_lr = r2_score(y_test, y_pred_lr)

print(f'MAE:  {mae_lr:.2f}')
print(f'RMSE: {rmse_lr:.2f}')
print(f'R²:   {r2_lr:.4f}')

results['Linear Regression'] = {'MAE': mae_lr, 'RMSE': rmse_lr, 'R²': r2_lr, 'Model': lr}

In [ ]:
# Model 2: Decision Tree Regressor
print('=' * 60)
print('MODEL 2: DECISION TREE REGRESSOR')
print('=' * 60)

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train_processed, y_train)
y_pred_dt = dt.predict(X_test_processed)

mae_dt = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = float(np.sqrt(mean_squared_error(y_test, y_pred_dt)))
r2_dt = r2_score(y_test, y_pred_dt)

print(f'MAE:  {mae_dt:.2f}')
print(f'RMSE: {rmse_dt:.2f}')
print(f'R²:   {r2_dt:.4f}')

results['Decision Tree'] = {'MAE': mae_dt, 'RMSE': rmse_dt, 'R²': r2_dt, 'Model': dt}

In [ ]:
# Model 3: Random Forest Regressor
print('=' * 60)
print('MODEL 3: RANDOM FOREST REGRESSOR')
print('=' * 60)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_processed, y_train)
y_pred_rf = rf.predict(X_test_processed)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = float(np.sqrt(mean_squared_error(y_test, y_pred_rf)))
r2_rf = r2_score(y_test, y_pred_rf)

print(f'MAE:  {mae_rf:.2f}')
print(f'RMSE: {rmse_rf:.2f}')
print(f'R²:   {r2_rf:.4f}')

results['Random Forest'] = {'MAE': mae_rf, 'RMSE': rmse_rf, 'R²': r2_rf, 'Model': rf}

---
## 8. Model Comparison

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.drop(columns=['Model'])
comparison_df = comparison_df.sort_values('R²', ascending=False)

print('=== MODEL COMPARISON (Ranked by R²) ===')
print(comparison_df.to_string())

# Highlight best model
best_model_name = comparison_df.index[0]
best_r2 = comparison_df.iloc[0]['R²']
best_rmse = comparison_df.iloc[0]['RMSE']
best_mae = comparison_df.iloc[0]['MAE']

print(f'\n' + '=' * 60)
print(f'BEST MODEL: {best_model_name}')
print(f'   R²:   {best_r2:.4f}')
print(f'   RMSE: {best_rmse:.2f}')
print(f'   MAE:  {best_mae:.2f}')
print('=' * 60)

---
## 9. Pipeline

Creating a complete `sklearn.Pipeline` that integrates:
- `ColumnTransformer` with `OneHotEncoder`, `OrdinalEncoder`, and `StandardScaler`
- The best performing regression model

This ensures preprocessing and model training happen together — no manual steps, no data leakage.

In [ ]:
from sklearn.pipeline import Pipeline

# Recreate fresh preprocessor for pipeline
preprocessor_pipe = ColumnTransformer(
    transformers=[
        ('nominal', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), nominal_cols),
        ('ordinal', OrdinalEncoder(), ordinal_cols),
        ('numerical', StandardScaler(), numerical_cols)
    ]
)

# Select best model instance based on comparison
if best_model_name == 'Random Forest':
    best_model_instance = RandomForestRegressor(n_estimators=100, random_state=42)
elif best_model_name == 'Decision Tree':
    best_model_instance = DecisionTreeRegressor(random_state=42)
else:
    best_model_instance = LinearRegression()

# Create full pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor_pipe),
    ('regressor', best_model_instance)
])

print('=== PIPELINE CREATED ===')
print(pipeline)

In [ ]:
# Fit the pipeline on training data
pipeline.fit(X_train, y_train)

# Predict on test data
y_pred_pipeline = pipeline.predict(X_test)

# Evaluate
mae_pipe = mean_absolute_error(y_test, y_pred_pipeline)
rmse_pipe = float(np.sqrt(mean_squared_error(y_test, y_pred_pipeline)))
r2_pipe = r2_score(y_test, y_pred_pipeline)

print('=== PIPELINE EVALUATION ===')
print(f'MAE:  {mae_pipe:.2f}')
print(f'RMSE: {rmse_pipe:.2f}')
print(f'R²:   {r2_pipe:.4f}')

---
## 10. Cross Validation

Cross-validation provides a more robust estimate of model performance by training/evaluating on multiple folds. k=5 means 80/20 splits repeated 5 times.

In [ ]:
from sklearn.model_selection import cross_val_score, cross_validate

# Perform 5-fold cross validation on the pipeline
cv_scores = cross_val_score(
    pipeline,
    X_train, y_train,
    cv=5,
    scoring='r2'
)

print('=== 5-FOLD CROSS VALIDATION (R²) ===')
for i, score in enumerate(cv_scores, 1):
    print(f'Fold {i}: {score:.4f}')

print(f'\nMean R²:       {cv_scores.mean():.4f}')
print(f'Std Dev R²:    {cv_scores.std():.4f}')

In [ ]:
# Also compute MAE and RMSE via cross_validate for more information
cv_full = cross_validate(
    pipeline,
    X_train, y_train,
    cv=5,
    scoring={'r2': 'r2', 'neg_mae': 'neg_mean_absolute_error', 'neg_rmse': 'neg_root_mean_squared_error'}
)

print('=== CROSS VALIDATION DETAILS ===')
print(f"R² scores per fold:  {[f'{s:.4f}' for s in cv_full['test_r2']]}")
print(f"R² Mean +- Std:       {cv_full['test_r2'].mean():.4f} +- {cv_full['test_r2'].std():.4f}")
print(f"MAE Mean +- Std:     {-cv_full['test_neg_mae'].mean():.2f} +- {cv_full['test_neg_mae'].std():.2f}")
print(f"RMSE Mean +- Std:    {-cv_full['test_neg_rmse'].mean():.2f} +- {cv_full['test_neg_rmse'].std():.2f}")

---
## 11. Hyperparameter Tuning

Using `GridSearchCV` to tune at least 2 hyperparameters of the best model.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define hyperparameter grid based on best model type
if isinstance(best_model_instance, RandomForestRegressor):
    param_grid = {
        'regressor__n_estimators': [50, 100, 200],
        'regressor__max_depth': [None, 10, 20],
        'regressor__min_samples_split': [2, 5, 10]
    }
elif isinstance(best_model_instance, DecisionTreeRegressor):
    param_grid = {
        'regressor__max_depth': [5, 10, 15, None],
        'regressor__min_samples_split': [2, 5, 10],
        'regressor__min_samples_leaf': [1, 2, 4]
    }
else:
    param_grid = {
        'regressor__fit_intercept': [True, False]
    }

print('=== HYPERPARAMETER GRID ===')
print(param_grid)

In [ ]:
# GridSearchCV
grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print('\n' + '=' * 60)
print('GRID SEARCH RESULTS')
print('=' * 60)
print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV R²:      {grid_search.best_score_:.4f}')
print(f'Best Estimator:  {grid_search.best_estimator_}')

---
## 12. Final Evaluation

Evaluate the tuned model on the **held-out test set**.

In [ ]:
# Predict on test set with tuned model
y_pred_tuned = grid_search.best_estimator_.predict(X_test)

# Calculate metrics
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = float(np.sqrt(mean_squared_error(y_test, y_pred_tuned)))
r2_tuned = r2_score(y_test, y_pred_tuned)

print('=' * 60)
print('FINAL EVALUATION - TUNED MODEL ON TEST SET')
print('=' * 60)
print(f'Mean Absolute Error (MAE):  {mae_tuned:.2f}')
print(f'Root Mean Squared Error:    {rmse_tuned:.2f}')
print(f'R² Score:                   {r2_tuned:.4f}')
print('=' * 60)

In [ ]:
# Compare tuned vs untuned performance
print('=== PERFORMANCE COMPARISON: UNTUNED vs TUNED ===')
comparison_final = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Untuned': [mae_pipe, rmse_pipe, r2_pipe],
    'Tuned': [mae_tuned, rmse_tuned, r2_tuned]
})
print(comparison_final.to_string(index=False))

---
## 13. Model Recommendation

In [ ]:
print('=' * 80)
print('MODEL RECOMMENDATION')
print('=' * 80)

best_model_final = grid_search.best_estimator_.named_steps['regressor'].__class__.__name__

print(f'''
Recommended Model: {best_model_final}

Why this model was selected:
  - Achieved the highest R² score among all models tested.
  - Random Forest handles non-linear relationships well, which is critical for social media
    data where interactions follow complex, non-linear patterns.
  - Ensemble method reduces overfitting compared to a single Decision Tree.
  - Robust to outliers and skewed distributions (common in engagement metrics).
  - Feature importance scores provide interpretability.

Advantages:
  - No feature scaling needed internally (trees are scale-invariant).
  - Captures feature interactions automatically.
  - Handles mixed data types (numerical + categorical) well.
  - Built-in feature importance for understanding drivers of engagement.

Limitations:
  - Model size grows with number of trees (can be memory-heavy).
  - Less interpretable than Linear Regression (no simple coefficients).
  - May overfit on small datasets if hyperparameters are not tuned.
  - Cannot extrapolate beyond training range (unlike Linear Regression).

Business Usefulness:
  - Social media teams can predict engagement before posting.
  - Identify which post characteristics drive the most interactions.
  - Optimize content strategy: best post type, timing, and category.
  - Data-driven decisions for paid promotion investments.
  - Forecast ROI of social media campaigns.
''')

# Feature importance (if available)
if hasattr(grid_search.best_estimator_.named_steps['regressor'], 'feature_importances_'):
    importances = grid_search.best_estimator_.named_steps['regressor'].feature_importances_
    
    # Feature importance dataframe
    feat_imp_df = pd.DataFrame({
        'Feature': all_feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print('\n=== FEATURE IMPORTANCE ===')
    print(feat_imp_df.to_string(index=False))

---
## 14. README.md (Generated Below)

In [ ]:
# Generate README.md content
readme_content = f'''# Predictive Modeling Pipeline - Facebook Performance Metrics

## Project Overview

This project builds a complete machine learning pipeline to predict Facebook post engagement (Total Interactions) for a renowned cosmetics brand. The pipeline includes data cleaning, feature engineering, model training, hyperparameter tuning, and evaluation.

**Problem Type:** Regression  
**Target Variable:** Total Interactions (sum of likes + comments + shares)  
**Best Model:** {best_model_final}  
**Test R² Score:** {r2_tuned:.4f}

---

## Business Problem

A cosmetics brand wants to predict how well a Facebook post will perform before publishing it. By forecasting Total Interactions, the social media team can:
- Choose the best post type (Photo, Status, Video, Link)
- Determine optimal posting time (month, weekday, hour)
- Decide whether paid promotion is worthwhile
- Maximize audience engagement and brand visibility

## Dataset Description

- **Source:** Facebook posts from a renowned cosmetics brand (2014)
- **Instances:** 500 posts
- **Features:** 19 attributes (7 pre-publication + 12 post-impact metrics)
- **Citation:** Moro et al. (2016) - Journal of Business Research

## Target

**Total Interactions** = comment + like + share

| Statistic | Value |
|-----------|-------|
| Mean | {y.mean():.2f} |
| Median | {y.median():.2f} |
| Min | {y.min() } |
| Max | {y.max() } |
| Std | {y.std():.2f} |

## Features

| Feature | Type | Description |
|---------|------|-------------|
| Page total likes | Numerical | Number of page fans at posting |
| Type | Nominal | Photo, Status, Video, Link |
| Category | Ordinal | 1, 2, or 3 |
| Post Month | Numerical | Month of the year (1-12) |
| Post Weekday | Numerical | Day of week (1-7) |
| Post Hour | Numerical | Hour of day (0-23) |
| Paid | Binary | Whether post was promoted |
| Page_Likes_Category | Ordinal | Binned: Small, Medium, Large |

## Preprocessing

1. **Missing Values:** Imputed comment/like/share with 0, Post Hour with median
2. **Categorical Creation:** Binned Page total likes into Small/Medium/Large
3. **Train-Test Split:** 80/20 stratified by Type (before encoding to prevent leakage)
4. **Encoding:** OneHotEncoder for Type (nominal), OrdinalEncoder for Category and Page_Likes_Category (ordinal)
5. **Scaling:** StandardScaler for numerical features (fit on training data only)

### Encoding Justification

- **OneHotEncoder:** Type is nominal (no ordering). One-hot avoids implying false ordinal relationships (e.g., Photo > Status > Video).
- **OrdinalEncoder:** Category (1,2,3) and Page_Likes_Category (Small < Medium < Large) have natural ordering, so ordinal encoding preserves this structure efficiently.

### Scaling Justification

StandardScaler centers features to mean=0 and scales to unit variance. This is critical when features have vastly different scales (Page total likes ~100,000 vs Post Hour ~12). Linear Regression and tree-based models benefit from standardized inputs for stable convergence and fair feature importance.

## Models Compared

| Model | MAE | RMSE | R² |
|-------|-----|------|----|
| Linear Regression | {results['Linear Regression']['MAE']:.2f} | {results['Linear Regression']['RMSE']:.2f} | {results['Linear Regression']['R²']:.4f} |
| Decision Tree | {results['Decision Tree']['MAE']:.2f} | {results['Decision Tree']['RMSE']:.2f} | {results['Decision Tree']['R²']:.4f} |
| Random Forest | {results['Random Forest']['MAE']:.2f} | {results['Random Forest']['RMSE']:.2f} | {results['Random Forest']['R²']:.4f} |

## Evaluation Metrics

- **MAE** - Mean Absolute Error (interpretable in same units)
- **RMSE** - Root Mean Squared Error (penalizes large errors)
- **R²** - Coefficient of Determination (variance explained)

## Cross Validation Results

5-fold cross-validation on the {best_model_name} model:

| Metric | Mean +/- Std |
|--------|------------|
| R² | {cv_full['test_r2'].mean():.4f} +/- {cv_full['test_r2'].std():.4f} |
| MAE | {-cv_full['test_neg_mae'].mean():.2f} +/- {cv_full['test_neg_mae'].std():.2f} |
| RMSE | {-cv_full['test_neg_rmse'].mean():.2f} +/- {cv_full['test_neg_rmse'].std():.2f} |

## Hyperparameter Search

- **Method:** GridSearchCV with 5-fold CV
- **Parameters tuned:** {list(param_grid.keys())}
- **Best parameters:** {grid_search.best_params_}
- **Best CV R²:** {grid_search.best_score_:.4f}

## Final Recommendation

**{best_model_final}** is the recommended model because:
- Highest R² on test set ({r2_tuned:.4f})
- Captures non-linear patterns in engagement data
- Robust to outliers and mixed data types
- Provides feature importance for business insights

## Installation

```bash
pip install -r requirements.txt
```

## Requirements

See `requirements.txt` for all dependencies.

## How to Run

1. Clone the repository
2. Install dependencies: `pip install -r requirements.txt`
3. Open Jupyter Notebook: `jupyter notebook Predictive_Modeling.ipynb`
4. Run all cells sequentially

## Project Structure

```
├── Predictive_Modeling.ipynb  # Main notebook
├── dataset_Facebook.csv       # Dataset
├── README.md                  # This file
├── requirements.txt           # Dependencies
```
'''

# Save README with UTF-8 encoding to handle special characters
with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

print('README.md generated successfully!')

---
## 15. Generate requirements.txt

In [ ]:
# Generate requirements.txt automatically
import pkg_resources

# List of packages used in this notebook
required_packages = [
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'jupyter',
    'notebook'
]

# Get installed versions
installed = {pkg.key: pkg.version for pkg in pkg_resources.working_set}

with open('requirements.txt', 'w') as f:
    for pkg in required_packages:
        version = installed.get(pkg, '>=0.0.0')
        if version != '>=0.0.0':
            f.write(f'{pkg}=={version}\n')
        else:
            f.write(f'{pkg}\n')

print('requirements.txt generated successfully!')
print('\n=== REQUIREMENTS ===')
with open('requirements.txt', 'r') as f:
    print(f.read())

---
## 16. Repository Structure Summary

In [ ]:
print('=' * 80)
print('PROJECT STRUCTURE')
print('=' * 80)
print('''
part-2/
├── Predictive_Modeling.ipynb    <- Jupyter Notebook (this file)
├── dataset_Facebook.csv         <- Original dataset
├── README.md                    <- Project documentation
├── requirements.txt             <- Python dependencies
└── Facebook_metrics.txt         <- Dataset description
''')
print('All steps completed successfully!')